In [1]:
# import pydub
# 
# audio = pydub.AudioSegment.from_file('F:\Experiments\output_audio_2b_(00_01_30)_(00_06_30).flac')
# audio = audio.set_channels(1)
# print(audio.channels)
# y = np.array(audio.get_array_of_samples())
# # mono_streams = audio.split_to_mono()
# print(y.shape)
# y = y.reshape(-1, audio.channels) 
# y.shape
# # y = np.concatenate([np.array(a.get_array_of_samples()) for a in mono_streams])
# # y.shape
# # np.mean(y,axis=0).shape
# # audio.get_array_of_samples().shape
# # np.array(audio.get_array_of_samples()).shape


In [2]:
# import numpy as np
# import plotly.graph_objs as go
# from plotly.subplots import make_subplots
# 
# 
# import plotly.io as pio
# pio.renderers.default = 'browser'
# 
# 
# decimation_factor = 30
# 
# sample_rate = (44100/decimation_factor)  # 44.1 kHz sample rate
# 
# audio_dec_1 = decimate(audio_a, decimation_factor)
# audio_dec_2 = decimate(audio_b, decimation_factor)
# 
# duration = int(audio_dec_1.shape[0]/sample_rate)  # 2 # 2 seconds
# # 
# # t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)
# # 
# # # Create subplots with 2 rows and 1 column
# # fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05)
# # 
# # # Add the first audio signal to the first subplot
# # fig.add_trace(
# #     go.Scattergl(x=t, y=audio_dec_1, mode="lines", name="Audio Signal 1"),
# #     row=1, col=1
# # )
# # 
# # # Add the second audio signal (using scattergl for better performance)
# # fig.add_trace(
# #     go.Scattergl(x=t, y=audio_dec_2, mode="lines", name="Audio Signal 2"),
# #     row=2, col=1
# # )
# # 
# # # Update layout for better visualization
# # fig.update_layout(
# #     height=600,  # Height of the entire plot
# #     title_text="Two Audio Signals with Zoom (Downsampled and Optimized)",
# #     showlegend=False,
# #     xaxis_title="Time (seconds)",
# #     yaxis_title="Amplitude",
# # )
# # 
# # # Enable zoom and pan functionality
# # fig.update_xaxes(rangeslider_visible=True)
# # fig.update_layout(xaxis=dict(autorange=True))
# # 
# # # Show the figure
# # fig.show()


In [3]:
# import plotly.io as pio
# from plotly.subplots import make_subplots
# import plotly.graph_objs as go
# 
# pio.renderers.default = 'browser'
# 
# t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)
# 
# # Create subplots with 2 rows and 1 column
# fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05)
# 
# # Add the first audio signal to the first subplot
# fig.add_trace(
#     go.Scattergl(x=t, y=audio_dec_1, mode="lines", name="Audio Signal 1"),
#     row=1, col=1
# )
# 
# # Add the second audio signal (using scattergl for better performance)
# fig.add_trace(
#     go.Scattergl(x=t, y=audio_dec_2, mode="lines", name="Audio Signal 2"),
#     row=2, col=1
# )
# 
# # Update layout for better visualization
# fig.update_layout(
#     height=600,  # Height of the entire plot
#     title_text="Two Audio Signals with Zoom (Downsampled and Optimized)",
#     showlegend=False,
#     xaxis_title="Time (seconds)",
#     yaxis_title="Amplitude",
# )
# 
# # Enable zoom and pan functionality
# fig.update_xaxes(rangeslider_visible=True)
# fig.update_layout(xaxis=dict(autorange=True))
# 
# # Show the figure
# fig.show()

In [4]:
# max_abs = max(np.max(np.abs(audio_a)), np.max(np.abs(audio_a)))
# min_abs = min(np.max(np.abs(audio_a)), np.max(np.abs(audio_a)))
# 
# audio_a_normalized = normalize_by_rms(audio_a)  #* max_abs
# audio_b_normalized = normalize_by_rms(audio_b)  #* max_abs
# 
# summ = (audio_a_normalized + audio_b_normalized) / 2
# diff = (audio_a_normalized - audio_b_normalized) / 2
# 
# display_two_signals(summ, diff, 48000)
# # Step 2: Compute the cross-correlation

In [5]:
def pad_shorter_signal(signal1, signal2):
    max_len = max(len(signal1), len(signal2))

    # Pad both signals to the maximum length
    signal1_padded = np.pad(signal1, (0, max_len - len(signal1)), mode='constant')
    signal2_padded = np.pad(signal2, (0, max_len - len(signal2)), mode='constant')

    return max_len, signal1_padded, signal2_padded


def compute_shifted_correlations(signal_a: np.ndarray, signal_b: np.ndarray, max_shift: int):
    """
    Computes the correlation between signal_a and signal_b for multiple shifts.
    
    Args:
    signal_a: First signal (numpy array)
    signal_b: Second signal (numpy array)
    max_shift: Maximum number of shifts to apply (positive and negative)

    Returns:
    correlations: List of correlations for each shift.
    shifts: List of shift values corresponding to the correlations.
    """
    correlations = []
    shifts = []

    # Ensure signals are the same length
    if len(signal_a) != len(signal_b):
        raise ValueError("The signals must be of equal length.")

    n = len(signal_a)

    # Shift from -max_shift to +max_shift
    for shift in tqdm(range(-max_shift, max_shift + 1)):
        shifted_b = calculate_shift(signal_b, shift)

        # Compute the correlation (r2_score in this case) between signal_a and shifted signal_b
        correlation = r2_score(signal_a, shifted_b)
        correlations.append(correlation)
        shifts.append(shift)

    return correlations, shifts


def fft_cuda(signal_np):
    signal_cp = cp.array(signal_np)
    fft_result_cp = cp.fft.fft(signal_cp)
    fft_result_np = cp.asnumpy(fft_result_cp)
    return fft_result_np


def ifft_cuda(signal_np):
    signal_cp = cp.array(signal_np)
    fft_result_cp = cp.fft.ifft(signal_cp)
    fft_result_np = cp.asnumpy(fft_result_cp)
    return fft_result_np


def shifted_correlation_fft_noncuda(signal_np_1, signal_np_2):
    # Compute FFT of both signals
    fft_signal1 = fft(signal_np_1)  #  fft_cuda(signal_np_1)  # 
    fft_signal2 = fft(signal_np_2)  #  fft_cuda(signal_np_2) 

    # Compute cross-spectral density (element-wise multiplication)
    cross_spectrum = fft_signal1 * np.conj(fft_signal2)

    # Inverse FFT to get cross-correlation
    cross_correlation = ifft(cross_spectrum)
    return cross_correlation

    # Use fftshift to center the zero-lag correlation


def shifted_correlation_fft_cuda(signal_np_1, signal_np_2):
    cp.cuda.Device().synchronize()

    print("Calculating FFTs")
    signal_cp_1 = cp.array(signal_np_1, dtype=cp.float32)
    # print('signal_cp_1',signal_cp_1.shape)
    # RFFT: out : complex ndarray # If n is even, the length of the transformed axis is (n/2)+1
    fft_result_cp_1 = cp.fft.rfft(signal_cp_1)
    # print('fft_result_cp_1',fft_result_cp_1.shape)
    signal_cp_2 = cp.array(signal_np_2, dtype=cp.float32)
    fft_result_cp_2 = cp.fft.rfft(signal_cp_2)

    del signal_cp_1, signal_cp_2
    cp.cuda.Device().synchronize()

    print("Multiplying FFTs")
    cross_spectrum_cp = fft_result_cp_1 * cp.conj(fft_result_cp_2)
    # print('cross_spectrum_cp',cross_spectrum_cp.shape)

    del fft_result_cp_1, fft_result_cp_2
    cp.cuda.Device().synchronize()

    print("Calculating IFFT")
    cross_correlation = cp.fft.ifft(cross_spectrum_cp)
    # print('cross_correlation',cross_correlation.shape) 
    del cross_spectrum_cp
    cp.cuda.Device().synchronize()

    print("Converting to NumPy")
    cross_correlation_np = cp.asnumpy(cross_correlation.real)
    del cross_correlation
    cp.cuda.Device().synchronize()

    return cross_correlation_np  # The returned size is n/2+1 for original signal n


def shifted_correlation_fft(signal1, signal2, rate=1):
    # Subtract mean to make the signals zero-mean
    signal1 -= np.mean(signal1)
    signal2 -= np.mean(signal2)

    n, signal1, signals2 = pad_shorter_signal(signal1, signal2)
    pad_length = 2 * n

    # Zero-pad the signals to avoid circular convolution
    padded_signal1 = np.pad(signal1, (0, pad_length - n), 'constant')
    padded_signal2 = np.pad(signal2, (0, pad_length - n), 'constant')

    print(n, pad_length, padded_signal1.shape, padded_signal2.shape)

    # cross_correlation = shifted_correlation_fft_cuda(padded_signal1, padded_signal2)
    # The returned size is n/2+1 for original signal n, so in this case n+1 for 2n after padding
    cross_correlation = shifted_correlation_fft_cuda(padded_signal1, padded_signal2)
    print('np cross correlation shape', cross_correlation.shape)

    max_time = cross_correlation.shape[0] // 2
    shifts = range(0 - max_time, max_time + 1)
    print(min(shifts), max(shifts), max_time)
    shifted_cross_correlation = fftshift(cross_correlation)
    print('shifted cross correlation ', shifted_cross_correlation.shape)

    # Normalization: normalize by the square root of the product of signal powers (autocorrelations)
    norm_factor = np.sqrt(np.sum(np.abs(padded_signal1) ** 2) * np.sum(np.abs(padded_signal2) ** 2))  # 1
    shifted_cross_correlation_normalized = shifted_cross_correlation / norm_factor
    # Return the real part (cross-correlation is real-valued)
    correlations = np.real(shifted_cross_correlation_normalized)

    # shifts = range(-int(correlations_rate / 2), int(correlations_rate / 2))  # Might only work for even numbers
    shifts = [shift / rate for shift in shifts]
    print(min(shifts), max(shifts), len(shifts))
    return correlations, shifts

NameError: name 'np' is not defined

In [6]:
# correlations, shifts = compute_shifted_correlations(audio_a_slice, audio_b_slice, max_shift)

def calculate_best_correlation(audio_a_slice, audio_b_slice):
    correlations, shifts = shifted_correlation_fft(audio_a_slice, audio_b_slice, 24000)
    # 
    # Plot the correlations
    best_correlation_index = np.argmax(correlations)
    best_shift = shifts[best_correlation_index]
    best_corr = correlations[best_correlation_index]
    return best_shift, best_corr, best_correlation_index, correlations, shifts


audio_a_slice_mono = np.mean(audio_a_slice, axis=1)
audio_b_slice_mono = np.mean(audio_b_slice, axis=1)

best_shift, best_corr, best_correlation_index, correlations, shifts = \
    calculate_best_correlation(audio_a_slice_mono, audio_b_slice_mono)

print(best_shift, best_correlation_index, correlations.min(), correlations.max())

NameError: name 'np' is not defined

In [7]:
plot_correlations(shifts, correlations,
                  'Shifted Correlation using Fourier Transform', 'Lag (Shift)',
                  'Correlation')


NameError: name 'plot_correlations' is not defined

In [1]:
print("Hello")

Hello


In [ ]:
for i in range(10):
    print(i)
print("That's all folks")